In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [3]:
from src.data_loader import load_config, load_cell2cell_train, split_data
from src.pipeline_builder import build_full_pipeline, get_feature_names_after_preprocessing

In [4]:
config = load_config()

X, y = load_cell2cell_train(config)

print(f"Original Shape : {X.shape}")
print(f"Target Shape   : {y.shape}")

2026-06-26 22:21:23.280 | INFO     | src.data_loader:load_cell2cell_train:27 - Loading Cell2Cell training data...
2026-06-26 22:21:24.090 | INFO     | src.data_loader:load_cell2cell_train:33 - Loaded: (51047, 58)
2026-06-26 22:21:24.103 | INFO     | src.data_loader:load_cell2cell_train:42 - Churn rate: 0.288 (14711 / 51047 churners)
2026-06-26 22:21:24.140 | INFO     | src.data_loader:load_cell2cell_train:63 - After drops: (51047, 52)


Original Shape : (51047, 52)
Target Shape   : (51047,)


In [5]:
X_train, X_test, y_train, y_test = split_data(X, y, config)

print(f"Train Shape : {X_train.shape}")
print(f"Test Shape  : {X_test.shape}")
print(f"Train Churn Rate : {y_train.mean():.3f}")
print(f"Test Churn Rate  : {y_test.mean():.3f}")

2026-06-26 22:21:24.351 | INFO     | src.data_loader:split_data:260 - Train: (40837, 52) | Churn rate: 0.288
2026-06-26 22:21:24.354 | INFO     | src.data_loader:split_data:265 - Test: (10210, 52) | Churn rate: 0.288


Train Shape : (40837, 52)
Test Shape  : (10210, 52)
Train Churn Rate : 0.288
Test Churn Rate  : 0.288


In [6]:
pipeline = build_full_pipeline(
    LogisticRegression(max_iter=1000),
    X_train.iloc[:200],
    y_train.iloc[:200]
)

print(type(pipeline))
print("Pipeline Steps:", pipeline.named_steps.keys())

2026-06-26 22:21:24.383 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded ServiceArea: 119 levels
2026-06-26 22:21:24.391 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded Occupation: 8 levels
2026-06-26 22:21:24.394 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded PrizmCode: 4 levels
2026-06-26 22:21:24.399 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded CreditRating: 7 levels
2026-06-26 22:21:24.459 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (200, 85)
2026-06-26 22:21:24.461 | DEBUG    | src.pipeline_builder:build_preprocessor:128 - Preprocessor columns — num:48 ohe:18 ord:1 binary:15


<class 'sklearn.pipeline.Pipeline'>
Pipeline Steps: dict_keys(['fe', 'pre', 'clf'])


In [7]:
start = time.time()
pipeline.fit(X_train, y_train)
elapsed = time.time()-start

print(f"Training completed in {elapsed:.2f} seconds")

2026-06-26 22:21:24.539 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded ServiceArea: 729 levels


2026-06-26 22:21:24.553 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded Occupation: 8 levels
2026-06-26 22:21:24.562 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded PrizmCode: 4 levels
2026-06-26 22:21:24.575 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded CreditRating: 7 levels
2026-06-26 22:21:24.715 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (40837, 85)


Training completed in 10.48 seconds


In [8]:
fe = pipeline.named_steps["fe"]
X_fe = fe.transform(X_test)

print("FE Shape:", X_fe.shape)

expected_features = [
'credit_risk_score','device_diversity','dropped_rate','early_tenure',
'equipment_age_months','escalated_customer','handset_tier',
'has_overage','inactive_subs_ratio','is_family_plan',
'is_low_credit','log_care_calls','log_equipment_days',
'log_overage','loyal_customer','occupation_encoded',
'old_equipment','peak_ratio','premium_dissatisfied',
'premium_handset','prizm_encoded','retention_failed',
'retention_success_rate','revenue_declining',
'revenue_growing','revenue_per_minute',
'service_area_encoded','tenure_group',
'total_call_volume','total_complaint_calls',
'usage_declining','usage_declining_severe',
'very_old_equipment'
]

missing=[f for f in expected_features if f not in X_fe.columns]

print("Missing Engineered Features:", missing if missing else "None")

assert X_fe.shape[0]==X_test.shape[0]
assert len(missing)==0


2026-06-26 22:21:35.075 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (10210, 85)


FE Shape: (10210, 85)
Missing Engineered Features: None


In [9]:
pre = pipeline.named_steps["pre"]
X_pre = pre.transform(X_fe)

print("Preprocessed Shape:", X_pre.shape)

assert X_pre.shape[1] > X_fe.shape[1]
assert np.isnan(X_pre).sum()==0

print("✓ No NaN after preprocessing")


Preprocessed Shape: (10210, 92)
✓ No NaN after preprocessing


In [10]:
feature_names = get_feature_names_after_preprocessing(
    pipeline
)

print("Number of feature names :", len(feature_names))
print("Preprocessed shape      :", X_pre.shape)

print("Expected columns:", X_pre.shape[1])
print("Feature names   :", len(feature_names))



Number of feature names : 92
Preprocessed shape      : (10210, 92)
Expected columns: 92
Feature names   : 92


In [11]:
pred = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)

print("Prediction Shape:", pred.shape)
print("Probability Shape:", probs.shape)

assert probs.shape==(len(X_test),2)
assert probs.min()>=0
assert probs.max()<=1
assert np.isnan(probs).sum()==0
assert np.allclose(probs.sum(axis=1),1)

print("✓ Probability checks passed")

2026-06-26 22:21:35.440 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (10210, 85)
2026-06-26 22:21:35.772 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (10210, 85)


Prediction Shape: (10210,)
Probability Shape: (10210, 2)
✓ Probability checks passed


In [12]:
auc = roc_auc_score(y_test, probs[:,1])
acc = accuracy_score(y_test,pred)

print(f"ROC-AUC : {auc:.4f}")
print(f"Accuracy: {acc:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test,pred))

print("\nClassification Report")
print(classification_report(y_test,pred))

ROC-AUC : 0.6224
Accuracy: 0.7096

Confusion Matrix
[[7004  264]
 [2701  241]]

Classification Report
              precision    recall  f1-score   support

           0       0.72      0.96      0.83      7268
           1       0.48      0.08      0.14      2942

    accuracy                           0.71     10210
   macro avg       0.60      0.52      0.48     10210
weighted avg       0.65      0.71      0.63     10210



In [13]:
smote_pipeline = build_full_pipeline(
    LogisticRegression(max_iter=1000),
    X_train.iloc[:200],
    y_train.iloc[:200],
    use_smote=True
)

print(type(smote_pipeline))
print(smote_pipeline.named_steps.keys())

assert 'smote' in smote_pipeline.named_steps
print("✓ SMOTE pipeline verified")

2026-06-26 22:21:36.131 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded ServiceArea: 119 levels
2026-06-26 22:21:36.135 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded Occupation: 8 levels
2026-06-26 22:21:36.140 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded PrizmCode: 4 levels
2026-06-26 22:21:36.147 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded CreditRating: 7 levels
2026-06-26 22:21:36.208 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (200, 85)
2026-06-26 22:21:36.210 | DEBUG    | src.pipeline_builder:build_preprocessor:128 - Preprocessor columns — num:48 ohe:18 ord:1 binary:15


<class 'imblearn.pipeline.Pipeline'>
dict_keys(['fe', 'pre', 'smote', 'clf'])
✓ SMOTE pipeline verified


In [14]:
print(f"FE Output Shape    : {X_fe.shape}")
print(f"Preprocessed Shape : {X_pre.shape}")
print(f"Prediction Shape   : {pred.shape}")
print(f"Probability Shape  : {probs.shape}")
print(f"Feature Count      : {len(feature_names)}")
print(f"ROC-AUC            : {auc:.4f}")

FE Output Shape    : (10210, 85)
Preprocessed Shape : (10210, 92)
Prediction Shape   : (10210,)
Probability Shape  : (10210, 2)
Feature Count      : 92
ROC-AUC            : 0.6224
